<a href="https://colab.research.google.com/github/siphephelok/ActivityAndUseCaseDiagrams/blob/main/Email_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Importing Libraries and Data**

First, we need to ensure all the necessary data files (the `train` and `test` folders, and `train_labels.csv`) are accessible in your Colab environment. If they are not already in your Google Drive, please upload them to a location like `/content/drive/MyDrive/Email classifier/`.

In [ ]:
import pandas as pd
import nltk
from bs4 import BeautifulSoup ## Download necessary NLTK data
nltk.download('stopwords')
import nltk
import string
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup
import os
nltk.download('wordnet')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
# Defining the base path where the data is located.

base_path = '/content/drive/MyDrive/Email_Classifier/'

# Loading the train_labels.csv file
try:
    train_labels_df = pd.read_csv(os.path.join(base_path, 'train_labels.csv'))
    print('Successfully loaded train_labels.csv:')
    display(train_labels_df.head())
except FileNotFoundError:
    print(f"Error: 'train_labels.csv' not found at {os.path.join(base_path, 'train_labels.csv')}. Please ensure the file is uploaded and the path is correct.")
except Exception as e:
    print(f"An error occurred while loading train_labels.csv: {e}")

Successfully loaded train_labels.csv:


,filename,true_category
0,email_1.html,Account Management
1,email_2.html,Insurance Claims
2,email_3.html,Account Management
3,email_4.html,Investment Advisory
4,email_5.html,Investment Advisory


In [ ]:
# List HTML files in the 'train' folder
train_emails_path = os.path.join(base_path, 'train')
if os.path.isdir(train_emails_path):
    train_html_files = [f for f in os.listdir(train_emails_path) if f.endswith('.html')]
    print(f"Found {len(train_html_files)} HTML files in the 'train' folder. Showing first 5:")
    display(train_html_files[:5])
else:
    print(f"Error: 'train' folder not found at {train_emails_path}. Please ensure the folder is uploaded and the path is correct.")

# List HTML files in the 'test' folder
test_emails_path = os.path.join(base_path, 'test')
if os.path.isdir(test_emails_path):
    test_html_files = [f for f in os.listdir(test_emails_path) if f.endswith('.html')]
    print(f"Found {len(test_html_files)} HTML files in the 'test' folder. Showing first 5:")
    display(test_html_files[:5])
else:
    print(f"Error: 'test' folder not found at {test_emails_path}. Please ensure the folder is uploaded and the path is correct.")

Found 44 HTML files in the 'train' folder. Showing first 5:


['email_31.html',
 'email_26.html',
 'email_10.html',
 'email_2.html',
 'email_19.html']

Found 12 HTML files in the 'test' folder. Showing first 5:


['email_9.html',
 'email_2.html',
 'email_5.html',
 'email_1.html',
 'email_12.html']

In [ ]:
def read_html_file(file_path):
    """Reads an HTML file and returns its content."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

def extract_email_data(file_list, folder_path):
    """Extracts email content, metadata, and IDs from a list of HTML files."""
    emails_data = []
    for filename in file_list:
        file_id = filename.split('.')[0]
        file_path = os.path.join(folder_path, filename)
        raw_content = read_html_file(file_path)

        if raw_content:
            soup = BeautifulSoup(raw_content, 'html.parser')

            # Extract email body text
            email_body_div = soup.find('div', class_='email-body')
            clean_text = email_body_div.get_text(separator=' ', strip=True) if email_body_div else ''

            # Extract metadata from the 'meta' div
            email_id = None
            subject = None
            sender = None
            date_received = None

            meta_div = soup.find('div', class_='meta')
            if meta_div:
                email_id_tag = meta_div.find('div', {'data-field': 'email_id'})
                if email_id_tag: email_id = email_id_tag.get_text(strip=True)

                subject_tag = meta_div.find('div', {'data-field': 'subject'})
                if subject_tag: subject = subject_tag.get_text(strip=True)

                sender_tag = meta_div.find('div', {'data-field': 'sender'})
                if sender_tag: sender = sender_tag.get_text(strip=True)

                date_received_tag = meta_div.find('div', {'data-field': 'date_received'})
                if date_received_tag: date_received = date_received_tag.get_text(strip=True)

            emails_data.append({
                'id': file_id,
                'email_id': email_id,
                'subject': subject,
                'sender': sender,
                'date_received': date_received,
                'email_content_clean': clean_text, # Cleaned text content
                'email_content_raw': raw_content # Keep raw content for reference if needed
            })

    return pd.DataFrame(emails_data)

# Ensure train_labels_df is loaded (from previous steps)
if 'train_labels_df' not in locals():
    print("Loading train_labels_df as it was not found in current scope.")
    try:
        train_labels_df = pd.read_csv(os.path.join(base_path, 'train_labels.csv'))
    except FileNotFoundError:
        print(f"Error: 'train_labels.csv' not found at {os.path.join(base_path, 'train_labels.csv')}. Please ensure the file is uploaded and the path is correct.")
        train_labels_df = pd.DataFrame() # Create an empty DataFrame to avoid errors
    except Exception as e:
        print(f"An error occurred while loading train_labels.csv: {e}")
        train_labels_df = pd.DataFrame() # Create an empty DataFrame to avoid errors


# Processing training emails
print("Processing training emails...")
train_emails_df = extract_email_data(train_html_files, train_emails_path)

# Merging training emails with labels
# Extracting 'id' from 'filename' in train_labels_df to match 'id' from email filenames
train_labels_df['id'] = train_labels_df['filename'].apply(lambda x: x.split('.')[0]).astype(str)
train_df_combined = pd.merge(train_emails_df, train_labels_df, on='id', how='left')
print("Combined training data head:")
display(train_df_combined.head())
print(f"Combined training data shape: {train_df_combined.shape}")

# Processing test emails (no labels for test set)
print("Processing test emails...")
test_df_combined = extract_email_data(test_html_files, test_emails_path)
print("Combined test data head:")
display(test_df_combined.head())
print(f"Combined test data shape: {test_df_combined.shape}")

Processing training emails...
Combined training data head:


,id,email_id,subject,sender,date_received,email_content_clean,email_content_raw,filename,true_category
0,email_31,32,Market Volatility Concerns,rbrown@gmail.com,2025-07-03,"Market Concerns Dear Financial Advisor, Given ...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",email_31.html,Investment Advisory
1,email_26,25,Regulatory Update,mary.t@hotmail.com,2025-06-07,"Compliance Notice Dear Clients, Please be awar...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",email_26.html,Other
2,email_10,20,Job Application Response,security@redrock.com,2025-03-18,"Employment Dear Applicant, Thank you for your ...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",email_10.html,Other
3,email_2,24,Disability Insurance Claim,pwhite@hotmail.com,2025-08-27,"Disability Claim Dear Claims Team, I need to f...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",email_2.html,Insurance Claims
4,email_19,47,Account Alerts Setup,lgarcia@yahoo.com,2025-01-22,"Account Alerts Dear Customer Service, I would ...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",email_19.html,Account Management


Combined training data shape: (44, 9)
Processing test emails...
Combined test data head:


,id,email_id,subject,sender,date_received,email_content_clean,email_content_raw
0,email_9,48,Market Research Request,gphillips@gmail.com,2025-08-31,"Market Research Dear Research Team, Could you ...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ..."
1,email_2,44,Investment Performance Report,lwilson@hotmail.com,2025-06-04,"Performance Report Dear Financial Team, Could ...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ..."
2,email_5,9,Health Insurance Claim,chris.lee@outlook.com,2025-03-18,"Health Claim Dear Claims Department, I need to...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ..."
3,email_1,35,Financial Education Workshop,hr@redrock.com,2025-02-24,"Education Event Dear Clients, Join us for our ...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ..."
4,email_12,41,Account Fee Inquiry,ajackson@yahoo.com,2025-02-09,"Fee Inquiry Dear Customer Service, I noticed a...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ..."


Combined test data shape: (12, 7)


In [ ]:
# Saving the combined training data to a CSV file
train_output_path = os.path.join(base_path, 'train_emails_with_labels.csv')
train_df_combined.to_csv(train_output_path, index=False)
print(f"Combined training data saved to: {train_output_path}")

# Saving the combined test data to a CSV file
test_output_path = os.path.join(base_path, 'test_emails.csv')
test_df_combined.to_csv(test_output_path, index=False)
print(f"Combined test data saved to: {test_output_path}")

Combined training data saved to: /content/drive/MyDrive/Email_Classifier/train_emails_with_labels.csv
Combined test data saved to: /content/drive/MyDrive/Email_Classifier/test_emails.csv


In [ ]:
train_df_combined.loc[1,"email_content_clean"]

'Compliance Notice Dear Clients, Please be aware of new regulatory requirements that may affect your accounts. We will be sending detailed information shortly. RedRock Compliance Department'

In [ ]:
del train_df_combined['id']
del train_df_combined['filename']
del train_df_combined['email_content_raw']

In [ ]:
train_df_combined

,email_id,subject,sender,date_received,email_content_clean,true_category
0,32,Market Volatility Concerns,rbrown@gmail.com,2025-07-03,"Market Concerns Dear Financial Advisor, Given ...",Investment Advisory
1,25,Regulatory Update,mary.t@hotmail.com,2025-06-07,"Compliance Notice Dear Clients, Please be awar...",Other
2,20,Job Application Response,security@redrock.com,2025-03-18,"Employment Dear Applicant, Thank you for your ...",Other
3,24,Disability Insurance Claim,pwhite@hotmail.com,2025-08-27,"Disability Claim Dear Claims Team, I need to f...",Insurance Claims
4,47,Account Alerts Setup,lgarcia@yahoo.com,2025-01-22,"Account Alerts Dear Customer Service, I would ...",Account Management
5,19,Life Insurance Claim,rebecca.t@yahoo.com,2025-03-04,"Life Insurance Dear Claims Department, I need ...",Insurance Claims
6,39,Umbrella Insurance Claim,susan.c@hotmail.com,2025-04-19,"Umbrella Insurance Dear Claims Department, I n...",Insurance Claims
7,46,Dividend Reinvestment Plan,info@redrock.com,2025-05-07,"DRIP Inquiry Dear Investment Services, I am in...",Investment Advisory
8,4,Insurance Claim Submission,arichardson@hotmail.com,2025-02-25,"Insurance Claim To Whom It May Concern, I need...",Insurance Claims
9,26,Wire Transfer Request,sarah.j@yahoo.com,2025-06-21,"Wire Transfer Dear Wire Transfer Department, I...",Account Management


# **Data Cleaning**

In [ ]:
train_df=train_df_combined.copy()
test_df=test_df_combined.copy()

In [ ]:
#Checking missing values
train_df.isnull().sum()

,0
email_id,0
subject,0
sender,0
date_received,0
email_content_clean,0
true_category,0


In [ ]:
test_df.isnull().sum()

,0
id,0
email_id,0
subject,0
sender,0
date_received,0
email_content_clean,0
email_content_raw,0


In [ ]:
#Check for duplicates
train_df.duplicated().sum()

np.int64(0)

In [ ]:
test_df.duplicated().sum()

np.int64(0)

In [ ]:
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Converting to lowercase
    text = text.lower()

    # Removing any remaining HTML tags (should be mostly handled by BeautifulSoup, but as a safeguard)
    text = BeautifulSoup(text, 'html.parser').get_text(separator=' ', strip=True)

    # Removing numbers and special characters, keeping only alphabetic characters and spaces
    text = re.sub(r'[^a-z\s]', '', text) # Keep only lowercase letters and spaces

    # Tokenize
    tokens = text.split()

    # Removing stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return ' '.join(tokens)

print("Applying enhanced text cleaning to training data...")
train_df['cleaned_email_content'] = train_df['email_content_clean'].apply(clean_text)
print("Applying enhanced text cleaning to test data...")
test_df['cleaned_email_content'] = test_df['email_content_clean'].apply(clean_text)

print("Original vs Cleaned email content (Training Data):")
display(train_df[['email_content_clean', 'cleaned_email_content']].head())

print("Original vs Cleaned email content (Test Data):")
display(test_df[['email_content_clean', 'cleaned_email_content']].head())

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Applying enhanced text cleaning to training data...
Applying enhanced text cleaning to test data...
Original vs Cleaned email content (Training Data):


,email_content_clean,cleaned_email_content
0,"Market Concerns Dear Financial Advisor, Given ...",market concern dear financial advisor given re...
1,"Compliance Notice Dear Clients, Please be awar...",compliance notice dear client please aware new...
2,"Employment Dear Applicant, Thank you for your ...",employment dear applicant thank interest finan...
3,"Disability Claim Dear Claims Team, I need to f...",disability claim dear claim team need file dis...
4,"Account Alerts Dear Customer Service, I would ...",account alert dear customer service would like...


Original vs Cleaned email content (Test Data):


,email_content_clean,cleaned_email_content
0,"Market Research Dear Research Team, Could you ...",market research dear research team could provi...
1,"Performance Report Dear Financial Team, Could ...",performance report dear financial team could p...
2,"Health Claim Dear Claims Department, I need to...",health claim dear claim department need submit...
3,"Education Event Dear Clients, Join us for our ...",education event dear client join u upcoming fi...
4,"Fee Inquiry Dear Customer Service, I noticed a...",fee inquiry dear customer service noticed mont...


In [ ]:
train_df

,email_id,subject,sender,date_received,email_content_clean,true_category,cleaned_email_content
0,32,Market Volatility Concerns,rbrown@gmail.com,2025-07-03,"Market Concerns Dear Financial Advisor, Given ...",Investment Advisory,market concern dear financial advisor given re...
1,25,Regulatory Update,mary.t@hotmail.com,2025-06-07,"Compliance Notice Dear Clients, Please be awar...",Other,compliance notice dear client please aware new...
2,20,Job Application Response,security@redrock.com,2025-03-18,"Employment Dear Applicant, Thank you for your ...",Other,employment dear applicant thank interest finan...
3,24,Disability Insurance Claim,pwhite@hotmail.com,2025-08-27,"Disability Claim Dear Claims Team, I need to f...",Insurance Claims,disability claim dear claim team need file dis...
4,47,Account Alerts Setup,lgarcia@yahoo.com,2025-01-22,"Account Alerts Dear Customer Service, I would ...",Account Management,account alert dear customer service would like...
5,19,Life Insurance Claim,rebecca.t@yahoo.com,2025-03-04,"Life Insurance Dear Claims Department, I need ...",Insurance Claims,life insurance dear claim department need file...
6,39,Umbrella Insurance Claim,susan.c@hotmail.com,2025-04-19,"Umbrella Insurance Dear Claims Department, I n...",Insurance Claims,umbrella insurance dear claim department need ...
7,46,Dividend Reinvestment Plan,info@redrock.com,2025-05-07,"DRIP Inquiry Dear Investment Services, I am in...",Investment Advisory,drip inquiry dear investment service intereste...
8,4,Insurance Claim Submission,arichardson@hotmail.com,2025-02-25,"Insurance Claim To Whom It May Concern, I need...",Insurance Claims,insurance claim may concern need file claim re...
9,26,Wire Transfer Request,sarah.j@yahoo.com,2025-06-21,"Wire Transfer Dear Wire Transfer Department, I...",Account Management,wire transfer dear wire transfer department ne...


In [ ]:
print("Applying enhanced text cleaning to test data's subject...")
test_df['subject_cleaned'] = test_df['subject'].apply(clean_text)
train_df['subject_cleaned'] = train_df['subject'].apply(clean_text)

Applying enhanced text cleaning to test data's subject...


In [ ]:
train_df.head(1)

,email_id,subject,sender,date_received,email_content_clean,true_category,cleaned_email_content,subject_cleaned
0,32,Market Volatility Concerns,rbrown@gmail.com,2025-07-03,"Market Concerns Dear Financial Advisor, Given ...",Investment Advisory,market concern dear financial advisor given re...,market volatility concern


In [ ]:
del train_df["subject"]
del train_df["email_content_clean"]

In [ ]:
train_df.head(1)

,email_id,sender,date_received,true_category,cleaned_email_content,subject_cleaned
0,32,rbrown@gmail.com,2025-07-03,Investment Advisory,market concern dear financial advisor given re...,market volatility concern


### **Encoding the Target Variable**

First, we'll encode the `true_category` column in the training data into numerical labels. This is a necessary step for machine learning models that expect numerical input for the target variable.

In [ ]:
# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the 'true_category' column in the training data
train_df['true_category_encoded'] = label_encoder.fit_transform(train_df['true_category'])

# Display the mapping of categories to encoded labels
print("Category to Encoded Label Mapping:")
for i, category in enumerate(label_encoder.classes_):
    print(f"{category}: {i}")

print("\nTraining data with encoded target variable (first 5 rows):")
display(train_df[['true_category', 'true_category_encoded']].head())

Category to Encoded Label Mapping:
Account Management: 0
Insurance Claims: 1
Investment Advisory: 2
Loan Processing: 3
Other: 4

Training data with encoded target variable (first 5 rows):


,true_category,true_category_encoded
0,Investment Advisory,2
1,Other,4
2,Other,4
3,Insurance Claims,1
4,Account Management,0


In [ ]:
train_df.head(1)

,email_id,sender,date_received,true_category,cleaned_email_content,subject_cleaned,true_category_encoded
0,32,rbrown@gmail.com,2025-07-03,Investment Advisory,market concern dear financial advisor given re...,market volatility concern,2


In [ ]:
del train_df["true_category"]

In [ ]:
train_df.head(1)

,email_id,sender,date_received,cleaned_email_content,subject_cleaned,true_category_encoded
0,32,rbrown@gmail.com,2025-07-03,market concern dear financial advisor given re...,market volatility concern,2


### **TF-IDF Vectorization for Text Features**

Next, we'll convert the cleaned text content from `cleaned_email_content` and `subject_cleaned` into numerical features using TF-IDF. TF-IDF reflects the importance of a word in a document relative to its importance in the entire corpus. We'll create separate TF-IDF features for the email content and subject, and then combine them.

In [ ]:
# Initialize TF-IDF Vectorizer for email content - TF-IDF for Term Frequency-Inverse Document Frequency
tfidf_vectorizer_content = TfidfVectorizer(max_features=5000) # Limiting features to 5000 for demonstration

# Fit and transform training email content
X_train_content_tfidf = tfidf_vectorizer_content.fit_transform(train_df['cleaned_email_content'])

# Transform test email content
X_test_content_tfidf = tfidf_vectorizer_content.transform(test_df['cleaned_email_content'])

# Initialize TF-IDF Vectorizer for subject
tfidf_vectorizer_subject = TfidfVectorizer(max_features=1000) # Limiting features to 1000 for demonstration

# Fit and transform training subject
X_train_subject_tfidf = tfidf_vectorizer_subject.fit_transform(train_df['subject_cleaned'])

# Transform test subject
X_test_subject_tfidf = tfidf_vectorizer_subject.transform(test_df['subject_cleaned'])

print(f"Shape of TF-IDF features for training content: {X_train_content_tfidf.shape}")
print(f"Shape of TF-IDF features for test content: {X_test_content_tfidf.shape}")
print(f"Shape of TF-IDF features for training subject: {X_train_subject_tfidf.shape}")
print(f"Shape of TF-IDF features for test subject: {X_test_subject_tfidf.shape}")



# Convert TF-IDF sparse matrices to dense DataFrames
train_content_tfidf_df = pd.DataFrame(X_train_content_tfidf.toarray(),
                                    columns=[f'content_tfidf_{i}' for i in range(X_train_content_tfidf.shape[1])])
train_subject_tfidf_df = pd.DataFrame(X_train_subject_tfidf.toarray(),
                                    columns=[f'subject_tfidf_{i}' for i in range(X_train_subject_tfidf.shape[1])])

test_content_tfidf_df = pd.DataFrame(X_test_content_tfidf.toarray(),
                                   columns=[f'content_tfidf_{i}' for i in range(X_test_content_tfidf.shape[1])])
test_subject_tfidf_df = pd.DataFrame(X_test_subject_tfidf.toarray(),
                                   columns=[f'subject_tfidf_{i}' for i in range(X_test_subject_tfidf.shape[1])])

# Concatenate the new TF-IDF feature DataFrames with the original DataFrames
# Make sure to reset index if they don't match, or use pd.concat with axis=1

# For train_df
# First, drop the original text columns if no longer needed
train_df_processed = train_df.drop(columns=['cleaned_email_content', 'subject_cleaned'], errors='ignore').reset_index(drop=True)
train_df = pd.concat([train_df_processed, train_content_tfidf_df, train_subject_tfidf_df], axis=1)

# For test_df
test_df_processed = test_df.drop(columns=['cleaned_email_content', 'subject_cleaned'], errors='ignore').reset_index(drop=True)
test_df = pd.concat([test_df_processed, test_content_tfidf_df, test_subject_tfidf_df], axis=1)

print("\nUpdated train_df with TF-IDF features (first 5 rows):")
display(train_df.head())
print(f"New shape of train_df: {train_df.shape}")

print("\nUpdated test_df with TF-IDF features (first 5 rows):")
display(test_df.head())
print(f"New shape of test_df: {test_df.shape}")

Shape of TF-IDF features for training content: (44, 341)
Shape of TF-IDF features for test content: (12, 341)
Shape of TF-IDF features for training subject: (44, 78)
Shape of TF-IDF features for test subject: (12, 78)

Updated train_df with TF-IDF features (first 5 rows):


,email_id,sender,date_received,true_category_encoded,content_tfidf_0,content_tfidf_1,content_tfidf_2,content_tfidf_3,content_tfidf_4,content_tfidf_5,...,subject_tfidf_68,subject_tfidf_69,subject_tfidf_70,subject_tfidf_71,subject_tfidf_72,subject_tfidf_73,subject_tfidf_74,subject_tfidf_75,subject_tfidf_76,subject_tfidf_77
0,32,rbrown@gmail.com,2025-07-03,2,0.0,0.0,0.000000,0.0,0.000000,0.266035,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.596277,0.0
1,25,mary.t@hotmail.com,2025-06-07,4,0.0,0.0,0.125955,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.63935,0.0,0.000000,0.0
2,20,security@redrock.com,2025-03-18,4,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0
3,24,pwhite@hotmail.com,2025-08-27,1,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0
4,47,lgarcia@yahoo.com,2025-01-22,0,0.0,0.0,0.222187,0.0,0.208741,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0


New shape of train_df: (44, 423)

Updated test_df with TF-IDF features (first 5 rows):


,id,email_id,subject,sender,date_received,email_content_clean,email_content_raw,content_tfidf_0,content_tfidf_1,content_tfidf_2,...,subject_tfidf_68,subject_tfidf_69,subject_tfidf_70,subject_tfidf_71,subject_tfidf_72,subject_tfidf_73,subject_tfidf_74,subject_tfidf_75,subject_tfidf_76,subject_tfidf_77
0,email_9,48,Market Research Request,gphillips@gmail.com,2025-08-31,"Market Research Dear Research Team, Could you ...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,email_2,44,Investment Performance Report,lwilson@hotmail.com,2025-06-04,"Performance Report Dear Financial Team, Could ...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,email_5,9,Health Insurance Claim,chris.lee@outlook.com,2025-03-18,"Health Claim Dear Claims Department, I need to...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,email_1,35,Financial Education Workshop,hr@redrock.com,2025-02-24,"Education Event Dear Clients, Join us for our ...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,email_12,41,Account Fee Inquiry,ajackson@yahoo.com,2025-02-09,"Fee Inquiry Dear Customer Service, I noticed a...","<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",0.0,0.0,0.133954,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


New shape of test_df: (12, 426)


In [ ]:
train_df.head(5)

,email_id,sender,date_received,true_category_encoded,content_tfidf_0,content_tfidf_1,content_tfidf_2,content_tfidf_3,content_tfidf_4,content_tfidf_5,...,subject_tfidf_68,subject_tfidf_69,subject_tfidf_70,subject_tfidf_71,subject_tfidf_72,subject_tfidf_73,subject_tfidf_74,subject_tfidf_75,subject_tfidf_76,subject_tfidf_77
0,32,rbrown@gmail.com,2025-07-03,2,0.0,0.0,0.000000,0.0,0.000000,0.266035,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.596277,0.0
1,25,mary.t@hotmail.com,2025-06-07,4,0.0,0.0,0.125955,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.63935,0.0,0.000000,0.0
2,20,security@redrock.com,2025-03-18,4,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0
3,24,pwhite@hotmail.com,2025-08-27,1,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0
4,47,lgarcia@yahoo.com,2025-01-22,0,0.0,0.0,0.222187,0.0,0.208741,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0


### **Encoding Categorical Features (`sender`)**

We will use `LabelEncoder` to convert the `sender` column, which contains categorical string data, into numerical representations. This is important for machine learning models that require numerical inputs.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize a LabelEncoder for the 'sender' column
sender_encoder = LabelEncoder()

# Combine sender columns from train and test to fit the encoder on all unique senders
all_senders = pd.concat([train_df['sender'], test_df['sender']], axis=0).unique()
sender_encoder.fit(all_senders)

# Transform the 'sender' column for both training and test data
train_df['sender_encoded'] = sender_encoder.transform(train_df['sender'])
test_df['sender_encoded'] = sender_encoder.transform(test_df['sender'])

print("Sender to Encoded Label Mapping:")
for i, sender in enumerate(sender_encoder.classes_):
    print(f"{sender}: {i}")

print("\nTraining data with encoded sender (first 5 rows):")
display(train_df[['sender', 'sender_encoded']].head())

print("\nTest data with encoded sender (first 5 rows):")
display(test_df[['sender', 'sender_encoded']].head())

Sender to Encoded Label Mapping:
ajackson@yahoo.com: 0
alexm@gmail.com: 1
amanda.f@yahoo.com: 2
amiller@yahoo.com: 3
anthony.s@gmail.com: 4
arichardson@hotmail.com: 5
bwilson@gmail.com: 6
chris.lee@outlook.com: 7
cjones@gmail.com: 8
compliance@redrock.com: 9
crodriguez@outlook.com: 10
daniel.r@gmail.com: 11
davidk@outlook.com: 12
emily.chen@yahoo.com: 13
eturner@hotmail.com: 14
feedback@redrock.com: 15
gphillips@gmail.com: 16
hcarter@hotmail.com: 17
hr@redrock.com: 18
info@redrock.com: 19
jamesg@gmail.com: 20
jcooper@outlook.com: 21
jen.lee@yahoo.com: 22
jessica.a@yahoo.com: 23
johnsmith1985@gmail.com: 24
jtaylor@outlook.com: 25
kmurphy@gmail.com: 26
kwhite@gmail.com: 27
lgarcia@yahoo.com: 28
lwilson@hotmail.com: 29
marketing@redrock.com: 30
markt@gmail.com: 31
mary.t@hotmail.com: 32
melissa.b@yahoo.com: 33
michellew@yahoo.com: 34
michelley@hotmail.com: 35
mikedavis@outlook.com: 36
nicolea@yahoo.com: 37
pwhite@hotmail.com: 38
rachel.g@yahoo.com: 39
rbrown@gmail.com: 40
rebecca.t@yahoo.

,sender,sender_encoded
0,rbrown@gmail.com,40
1,mary.t@hotmail.com,32
2,security@redrock.com,47
3,pwhite@hotmail.com,38
4,lgarcia@yahoo.com,28



Test data with encoded sender (first 5 rows):


,sender,sender_encoded
0,gphillips@gmail.com,16
1,lwilson@hotmail.com,29
2,chris.lee@outlook.com,7
3,hr@redrock.com,18
4,ajackson@yahoo.com,0


In [ ]:
train_df.head(5)

,email_id,sender,date_received,true_category_encoded,content_tfidf_0,content_tfidf_1,content_tfidf_2,content_tfidf_3,content_tfidf_4,content_tfidf_5,...,subject_tfidf_69,subject_tfidf_70,subject_tfidf_71,subject_tfidf_72,subject_tfidf_73,subject_tfidf_74,subject_tfidf_75,subject_tfidf_76,subject_tfidf_77,sender_encoded
0,32,rbrown@gmail.com,2025-07-03,2,0.0,0.0,0.000000,0.0,0.000000,0.266035,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.596277,0.0,40
1,25,mary.t@hotmail.com,2025-06-07,4,0.0,0.0,0.125955,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.63935,0.0,0.000000,0.0,32
2,20,security@redrock.com,2025-03-18,4,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0,47
3,24,pwhite@hotmail.com,2025-08-27,1,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0,38
4,47,lgarcia@yahoo.com,2025-01-22,0,0.0,0.0,0.222187,0.0,0.208741,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0,28


In [ ]:
del train_df["sender"]

### **Processing Temporal Features (`date_received`)**

The `date_received` column contains date strings. To extract meaningful numerical features, we'll first convert them to datetime objects and then extract features like the month and day of the week.

In [ ]:
import pandas as pd

# Convert 'date_received' to datetime objects
train_df['date_received'] = pd.to_datetime(train_df['date_received'])
test_df['date_received'] = pd.to_datetime(test_df['date_received'])

# Extract numerical features from 'date_received'
train_df['received_month'] = train_df['date_received'].dt.month
train_df['received_day_of_week'] = train_df['date_received'].dt.dayofweek # Monday=0, Sunday=6
train_df['received_year'] = train_df['date_received'].dt.year

test_df['received_month'] = test_df['date_received'].dt.month
test_df['received_day_of_week'] = test_df['date_received'].dt.dayofweek
test_df['received_year'] = test_df['date_received'].dt.year

print("Training data with new date features (first 5 rows):")
display(train_df[['date_received', 'received_month', 'received_day_of_week', 'received_year']].head())

print("\nTest data with new date features (first 5 rows):")
display(test_df[['date_received', 'received_month', 'received_day_of_week', 'received_year']].head())

Training data with new date features (first 5 rows):


,date_received,received_month,received_day_of_week,received_year
0,2025-07-03,7,3,2025
1,2025-06-07,6,5,2025
2,2025-03-18,3,1,2025
3,2025-08-27,8,2,2025
4,2025-01-22,1,2,2025



Test data with new date features (first 5 rows):


,date_received,received_month,received_day_of_week,received_year
0,2025-08-31,8,6,2025
1,2025-06-04,6,2,2025
2,2025-03-18,3,1,2025
3,2025-02-24,2,0,2025
4,2025-02-09,2,6,2025


In [ ]:
train_df.head()

,email_id,date_received,true_category_encoded,content_tfidf_0,content_tfidf_1,content_tfidf_2,content_tfidf_3,content_tfidf_4,content_tfidf_5,content_tfidf_6,...,subject_tfidf_72,subject_tfidf_73,subject_tfidf_74,subject_tfidf_75,subject_tfidf_76,subject_tfidf_77,sender_encoded,received_month,received_day_of_week,received_year
0,32,2025-07-03,2,0.0,0.0,0.000000,0.0,0.000000,0.266035,0.266035,...,0.0,0.0,0.00000,0.0,0.596277,0.0,40,7,3,2025
1,25,2025-06-07,4,0.0,0.0,0.125955,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.63935,0.0,0.000000,0.0,32,6,5,2025
2,20,2025-03-18,4,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.00000,0.0,0.000000,0.0,47,3,1,2025
3,24,2025-08-27,1,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.00000,0.0,0.000000,0.0,38,8,2,2025
4,47,2025-01-22,0,0.0,0.0,0.222187,0.0,0.208741,0.000000,0.000000,...,0.0,0.0,0.00000,0.0,0.000000,0.0,28,1,2,2025


In [ ]:
train_df["received_year"]

,received_year
0,2025
1,2025
2,2025
3,2025
4,2025
5,2025
6,2025
7,2025
8,2025
9,2025


In [ ]:
del train_df["date_received"]

### **Change data frame to a matrix form**



### **Creating Feature Matrices (X) and Target Vector (y)**

Now, let's create the final feature matrices (`X_train`, `X_test`) and the target vector (`y_train`) by combining all the processed features: TF-IDF for email content and subject, encoded sender, and extracted temporal features. We will use `hstack` from `scipy.sparse` to combine the sparse TF-IDF matrices with our dense numerical features.

In [ ]:
from scipy.sparse import hstack, csr_matrix

# Get the target variable for training
y_train = train_df['true_category_encoded']

# Define the numerical features to include from the DataFrames
numerical_features = ['sender_encoded', 'received_month', 'received_day_of_week', 'received_year']

# Extract these numerical features for training and test data
X_train_numerical = train_df[numerical_features].values
X_test_numerical = test_df[numerical_features].values

# Combine TF-IDF features with numerical features using hstack
# Convert dense numerical features to sparse matrix to ensure compatibility with hstack
X_train = hstack([X_train_content_tfidf, X_train_subject_tfidf, csr_matrix(X_train_numerical)])
X_test = hstack([X_test_content_tfidf, X_test_subject_tfidf, csr_matrix(X_test_numerical)])

print(f"Final shape of training features (X_train): {X_train.shape}")
print(f"Shape of training target (y_train): {y_train.shape}")
print(f"Final shape of test features (X_test): {X_test.shape}")

Final shape of training features (X_train): (44, 423)
Shape of training target (y_train): (44,)
Final shape of test features (X_test): (12, 423)


# Train the model

### **Training a Logistic Regression Model**

Given our prepared feature matrices (`X_train`, `X_test`) and target vector (`y_train`), we'll start by training a Logistic Regression model. Logistic Regression is a good baseline classifier for text data and handles sparse matrices efficiently.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Initialize the Logistic Regression model
# Set max_iter for convergence with sparse data, and solver for better performance
model = LogisticRegression(random_state=42, max_iter=500, solver='liblinear')

# Train the model
model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


### **Making Predictions and Evaluating the Model**

Now, let's use the trained model to make predictions on the training data and evaluate its performance. Since we don't have true labels for the test set, we'll evaluate on the training data to get an initial understanding of the model's performance.

In [ ]:
# Make predictions on the training set
y_train_pred = model.predict(X_train)

# Evaluate the model on the training set
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
print("\nClassification Report (Training Data):\n", classification_report(y_train, y_train_pred))

# Make predictions on the test set
y_test_pred = model.predict(X_test)

print("\nPredictions for the first 5 test emails:\n", y_test_pred[:5])

Training Accuracy: 1.0

Classification Report (Training Data):
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      1.00      1.00         7
           2       1.00      1.00      1.00        12
           3       1.00      1.00      1.00         6
           4       1.00      1.00      1.00         6

    accuracy                           1.00        44
   macro avg       1.00      1.00      1.00        44
weighted avg       1.00      1.00      1.00        44


Predictions for the first 5 test emails:
 [2 2 1 4 0]


In [ ]:
# Get email_id from the original test_df
email_ids = test_df['email_id']

# Map numerical predictions back to original category names
# The 'label_encoder' object was fitted earlier on 'true_category' from train_df
predicted_categories = label_encoder.inverse_transform(y_test_pred)

# Get confidence scores (probabilities) for each prediction
# model.predict_proba returns probabilities for all classes
confidence_scores_all_classes = model.predict_proba(X_test)

# Extract the probability of the *predicted* class for each email
# y_test_pred contains the index of the predicted class
confidence_scores = [confidence_scores_all_classes[i, y_test_pred[i]] for i in range(len(y_test_pred))]

# Create a DataFrame for the results
results_df = pd.DataFrame({
    'email_id': email_ids,
    'predicted_category': predicted_categories,
    'confidence_score': confidence_scores
})

# Define the output path for the CSV file
output_csv_path = os.path.join(base_path, 'email_classification_predictions.csv')

# Save the results to a CSV file
results_df.to_csv(output_csv_path, index=False)

print(f"Predictions saved to: {output_csv_path}")
print("\nFirst 5 rows of the predictions file:")
display(results_df.head())

Predictions saved to: /content/drive/MyDrive/Email_Classifier/email_classification_predictions.csv

First 5 rows of the predictions file:


,email_id,predicted_category,confidence_score
0,48,Investment Advisory,0.500507
1,44,Investment Advisory,0.611381
2,9,Insurance Claims,0.635641
3,35,Other,0.304657
4,41,Account Management,0.727149


# Saving results to MyDrive

In [ ]:
results_df.to_csv("/content/drive/MyDrive/Email_Classifier/results.csv")